In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.3 MB/s eta 0:00:00


In [2]:
# import libraries
import numpy as np
import joblib
from ultralytics import YOLO

from sklearn.preprocessing import StandardScaler

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
#load ml classifier
import joblib

base_path = "/content/drive/MyDrive/Colab Notebooks/Multimodal - G. Project/"

model = joblib.load(base_path + "ml_lr_classifier.pkl")
scaler = joblib.load(base_path + "ml_lrـscaler.pkl")
le = joblib.load(base_path + "ml_lrـlabel_encoder.pkl")

In [5]:
#load yolo model
from ultralytics import YOLO

yolo_path = base_path + "yolo_models/indoor_fire_smoke_yolov8n/weights/best.pt"
detector = YOLO(yolo_path)

In [6]:
def sensor_decision(sample):
    sample = np.array(sample).reshape(1, -1)

    if sample.shape[1] != 8:
        raise ValueError(f"Expected 8 sensor features, got {sample.shape[1]}")

    sample_scaled = scaler.transform(sample)
    sensor_pred = model.predict(sample_scaled)
    sensor_pred_label = le.inverse_transform(sensor_pred)[0]
    return sensor_pred_label

In [7]:
CLASS_NAMES = {0: "fire", 1: "smoke"}

def image_decision(result, conf_threshold=0.40):
    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        return {"decision": "none", "max_conf": 0.0, "detections": []}

    detections = []
    for box in boxes:
        cls_id = int(box.cls.item())
        conf = float(box.conf.item())
        label = CLASS_NAMES.get(cls_id, str(cls_id))

        if conf >= conf_threshold:
            detections.append((label, conf))

    detections.sort(key=lambda x: x[1], reverse=True)

    if not detections:
        return {"decision": "none", "max_conf": 0.0, "detections": []}

    fire_scores = [conf for label, conf in detections if label == "fire"]
    smoke_scores = [conf for label, conf in detections if label == "smoke"]

    if fire_scores:
        return {
            "decision": "fire",
            "max_conf": max(fire_scores),
            "detections": detections
        }
    elif smoke_scores:
        return {
            "decision": "smoke",
            "max_conf": max(smoke_scores),
            "detections": detections
        }
    else:
        return {
            "decision": "none",
            "max_conf": 0.0,
            "detections": []
        }

In [8]:
def multimodal_fusion(sensor_pred, image_result_dict):
    img_decision = image_result_dict.get("decision", "none")
    img_conf = image_result_dict.get("max_conf", 0.0)

    if sensor_pred == "Background":
        if img_decision == "none":
            return {
                "sensor_prediction": sensor_pred,
                "image_decision": img_decision,
                "image_confidence": img_conf,
                "final_status": "No Alert",
                "reason": "Sensor branch says normal background and image branch found no visible hazard."
            }
        elif img_decision == "smoke":
            return {
                "sensor_prediction": sensor_pred,
                "image_decision": img_decision,
                "image_confidence": img_conf,
                "final_status": "Medium Alert",
                "reason": "Sensor branch says background, but image branch detected smoke."
            }
        elif img_decision == "fire":
            return {
                "sensor_prediction": sensor_pred,
                "image_decision": img_decision,
                "image_confidence": img_conf,
                "final_status": "High Alert",
                "reason": "Sensor branch says background, but image branch detected fire."
            }

    elif sensor_pred == "Nuisance":
        if img_decision == "none":
            return {
                "sensor_prediction": sensor_pred,
                "image_decision": img_decision,
                "image_confidence": img_conf,
                "final_status": "Low Alert",
                "reason": "Sensor says nuisance and image branch found no visible hazard."
            }
        elif img_decision == "smoke":
            return {
                "sensor_prediction": sensor_pred,
                "image_decision": img_decision,
                "image_confidence": img_conf,
                "final_status": "Medium Alert",
                "reason": "Sensor says nuisance and image branch detected smoke."
            }
        elif img_decision == "fire":
            return {
                "sensor_prediction": sensor_pred,
                "image_decision": img_decision,
                "image_confidence": img_conf,
                "final_status": "High Alert",
                "reason": "Sensor says nuisance, but image branch detected fire."
            }

    elif sensor_pred == "Fire":
        if img_decision == "fire":
            return {
                "sensor_prediction": sensor_pred,
                "image_decision": img_decision,
                "image_confidence": img_conf,
                "final_status": "High Alert",
                "reason": "Sensor branch indicates fire and image branch confirmed visible fire."
            }
        elif img_decision == "smoke":
            return {
                "sensor_prediction": sensor_pred,
                "image_decision": img_decision,
                "image_confidence": img_conf,
                "final_status": "High Alert",
                "reason": "Sensor branch indicates fire and image branch confirmed smoke."
            }
        elif img_decision == "none":
            return {
                "sensor_prediction": sensor_pred,
                "image_decision": img_decision,
                "image_confidence": img_conf,
                "final_status": "Medium Alert",
                "reason": "Sensor branch indicates fire, but image branch found no visible confirmation."
            }

    return {
        "sensor_prediction": sensor_pred,
        "image_decision": img_decision,
        "image_confidence": img_conf,
        "final_status": "Unknown",
        "reason": "Unexpected sensor or image prediction."
    }

In [11]:
"""
# 1) sensor
sample_sensor = np.load("/content/drive/MyDrive/Colab Notebooks/Multimodal - G. Project/sample_sensor.npy")
sensor_label = sensor_decision(sample_sensor)

# 2) image
img_path = "/content/drive/MyDrive/Colab Notebooks/Multimodal - G. Project/Indoor Fire Smoke/test/images"  # حطي صورة
result = detector.predict(source=img_path, imgsz=640, conf=0.25, verbose=False)[0]
img_decision = image_verification_decision(result)

# 3) fusion
final = multimodal_fusion(sensor_label, img_decision)

print("Sensor:", sensor_label)
print("Image :", img_decision)
print("Final :", final)
"""
import glob

sample_sensor = np.load(base_path + "sample_sensor.npy")
sensor_label = sensor_decision(sample_sensor)

img_list = glob.glob(base_path + "Indoor Fire Smoke/test/images/*.jpg")
img_path = img_list[0]

result = detector.predict(source=img_path, imgsz=640, conf=0.25, verbose=False)[0]
img_decision = image_decision(result, conf_threshold=0.40)

final = multimodal_fusion(sensor_label, img_decision)

print("Image path:", img_path)
print("Sensor:", sensor_label)
print("Image :", img_decision)
print("Final :", final)

Image path: /content/drive/MyDrive/Colab Notebooks/Multimodal - G. Project/Indoor Fire Smoke/test/images/part2-trimmed_mp4-60_jpg.rf.5cce00fa3d69f627490ecbfae1a34a78.jpg
Sensor: Background
Image : {'decision': 'fire', 'max_conf': 0.6660302877426147, 'detections': [('fire', 0.6660302877426147), ('fire', 0.6196780800819397)]}
Final : {'sensor_prediction': 'Background', 'image_decision': 'fire', 'image_confidence': 0.6660302877426147, 'final_status': 'High Alert', 'reason': 'Sensor branch says background, but image branch detected fire.'}
